<a href="https://colab.research.google.com/github/katjanieberle/py_causal_modeling/blob/main/project2_supervised_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% [markdown]
# ## Bibliotheken importieren
# Dieser Abschnitt importiert alle notwendigen Python-Bibliotheken für die statistische Analyse und Visualisierung.

# %% [code]
import pandas as pd  # Für Datenmanipulation und -analyse (DataFrames)
import numpy as np   # Für numerische Operationen, insbesondere Array-Manipulation
import statsmodels.api as sm  # Erweiterte statistische Modelle (z.B. ANOVA-Tabelle)
import statsmodels.formula.api as smf # Für Statistikmodelle mit R-ähnlichen Formeln (OLS, GLM, etc.)
from statsmodels.stats.multicomp import pairwise_tukeyhsd # Für Post-hoc-Tests (Tukey HSD) nach ANOVA
from scipy import stats  # Für statistische Tests, z.B. Shapiro-Wilk Test
import matplotlib.pyplot as plt # Für die Erstellung statischer, interaktiver und animierter Visualisierungen
import seaborn as sns # Für anspruchsvolle und informative statistische Grafiken, basierend auf matplotlib

# %% [markdown]
# ## 1. Marketing-Daten laden
# Anstatt synthetische Daten zu generieren, laden wir einen öffentlich verfügbaren Datensatz, der reale Marketingausgaben und Verkaufszahlen enthält. Dies ermöglicht eine realistischere Anwendung der statistischen Modelle.

# %% [code]
print("--- 1. Marketing-Daten laden ---")

# URL zum öffentlichen "Advertising" Datensatz
# Dieser Datensatz enthält die Ausgaben für TV, Radio und Newspaper Werbung sowie die resultierenden Verkäufe.
url = 'https://raw.githubusercontent.com/AppliedStat/StatLearning/master/data/Advertising.csv'

# Den Datensatz direkt von der URL in einen Pandas DataFrame laden.
# `index_col=0` weist Pandas an, die erste Spalte als Index zu verwenden und nicht als reguläre Datenspalte.
try:
    df_marketing = pd.read_csv(url, index_col=0)
except Exception as e:
    # Fehlerbehandlung, falls der Datensatz nicht geladen werden kann (z.B. bei Netzwerkproblemen)
    print(f"Fehler beim Laden des Datensatzes von der URL: {e}")
    print("Bitte überprüfen Sie die Internetverbindung oder die URL.")
    # In einem realen Szenario könnte hier ein Fallback zu einem lokalen Datensatz erfolgen

# Die ersten 5 Zeilen des geladenen DataFrames anzeigen, um eine Übersicht der Daten zu erhalten.
print("Erste 5 Zeilen des geladenen Marketing-Datensatzes:")
print(df_marketing.head())
print("\n")

# %% [markdown]
# ## 2. Multiple Lineare Regression
# Die multiple lineare Regression wird verwendet, um die Beziehung zwischen einer abhängigen Variable (hier: `Sales`) und mehreren unabhängigen Variablen (hier: `TV`, `Radio`, `Newspaper`) zu modellieren. Ziel ist es, die Koeffizienten zu schätzen, die die Stärke und Richtung dieser Beziehungen angeben.

# %% [code]
print("--- 2. Multiple Lineare Regression ---")

# Modellaufbau mit der `statsmodels.formula.api.ols` Funktion.
# Die Formel `Sales ~ TV + Radio + Newspaper` definiert `Sales` als abhängige Variable
# und `TV`, `Radio`, `Newspaper` als unabhängige Variablen.
# `.fit()` führt die tatsächliche Regression durch und schätzt die Modellparameter.
model_mlr = smf.ols('Sales ~ TV + Radio + Newspaper', data=df_marketing).fit()

# Die `.summary()` Methode liefert eine umfassende Übersicht über die Regressionsergebnisse,
# einschließlich Koeffizienten, Standardfehlern, p-Werten, R-squared und F-Statistiken.
print("\nModellzusammenfassung der Multiplen Linearen Regression:")
print(model_mlr.summary())

# %% [markdown]
# ### Erklärung der Regressionsergebnisse
# Dieser Abschnitt erläutert die wichtigsten Metriken aus der Modellzusammenfassung der multiplen linearen Regression.

# %% [code]
print("\n--- Erklärung der Regressionsergebnisse ---")

# #### 2.1. Bestimmtheitsmaß (R-squared)
# R-squared ist ein Maß dafür, wie gut die unabhängigen Variablen die Varianz in der abhängigen Variablen erklären.
print("\nBestimmtheitsmaß (R-squared):")
print(f"  Das R-squared (hier: {model_mlr.rsquared:.3f}) gibt den Anteil der Varianz in der abhängigen Variable (Sales) an, der durch die unabhängigen Variablen (Marketing-Ausgaben) erklärt wird.")
print("  Ein Wert von 1 bedeutet, dass das Modell 100% der Varianz erklärt, während 0 bedeutet, dass es keine Varianz erklärt.")
print(f"  In unserem Fall erklärt das Modell {model_mlr.rsquared*100:.1f}% der Varianz der Verkäufe. Dies ist ein relativ hoher Wert, was auf eine gute Anpassung des Modells hindeutet.")

# #### 2.2. Korrigiertes Bestimmtheitsmaß (Adjusted R-squared)
# Adjusted R-squared ist eine modifizierte Version des R-squared, die die Anzahl der Prädiktoren und die Stichprobengröße berücksichtigt. Es ist nützlicher für den Vergleich von Modellen mit unterschiedlicher Anzahl von Prädiktoren, da es eine Strafe für das Hinzufügen unnötiger Variablen enthält.
print("\nKorrigiertes Bestimmtheitsmaß (Adjusted R-squared):")
print(f"  Das Adjusted R-squared (hier: {model_mlr.rsquared_adj:.3f}) ist eine modifizierte Version des R-squared, die die Anzahl der Prädiktoren im Modell und die Stichprobengröße berücksichtigt.")
print("  Es ist nützlicher, um Modelle mit unterschiedlicher Anzahl von Prädiktoren zu vergleichen, da es bestraft, wenn unnötige Prädiktoren hinzugefügt werden.")
print("  Es wird bevorzugt, um Overfitting zu vermeiden, da das R-squared immer steigt, wenn neue Variablen hinzugefügt werden, auch wenn diese keinen echten Erklärungswert haben. Der geringe Unterschied zwischen R-squared und Adjusted R-squared deutet darauf hin, dass alle Prädiktoren relevant sind.")

# #### 2.3. F-Test (Gesamtbedeutung des Modells)
# Der F-Test prüft die Hypothese, dass alle Regressionskoeffizienten der unabhängigen Variablen gleichzeitig Null sind. Ein signifikanter F-Test zeigt an, dass das Modell als Ganzes statistisch bedeutsam ist und mindestens eine der unabhängigen Variablen die abhängige Variable erklärt.
print("\nF-Test (Test auf Gesamtbedeutung des Modells):")
print(f"  Der F-Statistik (hier: {model_mlr.fvalue:.2f}) und sein p-Wert (Prob (F-statistic): {model_mlr.f_pvalue:.3f}) testen die Nullhypothese, dass alle Regressionskoeffizienten gleichzeitig Null sind.")
print("  Ein kleiner p-Wert (typischerweise < 0.05) weist darauf hin, dass das Modell insgesamt statistisch signifikant ist und mindestens eine der unabhängigen Variablen die abhängige Variable erklärt.")
print(f"  Da unser p-Wert ({model_mlr.f_pvalue:.3f}) sehr klein ist, können wir die Nullhypothese verwerfen und schlussfolgern, dass unser Modell statistisch signifikant ist. Die Marketingausgaben haben also einen signifikanten Einfluss auf die Verkäufe.")

# #### 2.4. t-Tests (Bedeutung einzelner Koeffizienten)
# Die t-Tests prüfen die Signifikanz jedes einzelnen Regressionskoeffizienten. Ein kleiner p-Wert für einen Koeffizienten bedeutet, dass diese spezifische unabhängige Variable einen signifikanten Beitrag zur Erklärung der abhängigen Variablen leistet, unter Berücksichtigung der anderen Prädiktoren im Modell.
print("\nt-Tests (Test auf Bedeutung einzelner Koeffizienten):")
print("  Für jeden unabhängigen Prädiktor (z.B. TV) gibt es einen t-Wert und einen entsprechenden p-Wert (|P>|t|). Diese testen die Hypothese, dass der jeweilige Koeffizient Null ist.")
print("  Ein kleiner p-Wert (typischerweise < 0.05) bedeutet, dass der entsprechende Prädiktor einen statistisch signifikanten Beitrag zur Erklärung der abhängigen Variable leistet, unter Kontrolle der anderen Prädiktoren.")
print("  Schlussfolgerung für unsere Daten:")
for param, p_val in model_mlr.pvalues.items():
    if param != 'Intercept': # Der Intercept ist der Achsenabschnitt und wird oft separat interpretiert
        print(f"  - Der Prädiktor '{param}' hat einen p-Wert von {p_val:.3f}. Ist dieser kleiner als 0.05, so ist er signifikant. Hier: {'Signifikant' if p_val < 0.05 else 'Nicht signifikant'}.")

# %% [markdown]
# ## 3. Varianzanalyse (ANOVA) und Kovarianzanalyse (ANCOVA)
# **ANOVA** (Analysis of Variance) wird verwendet, um zu testen, ob es signifikante Unterschiede zwischen den Mittelwerten von drei oder mehr Gruppen gibt. Sie basiert auf der Zerlegung der Gesamtvarianz in die Varianz zwischen den Gruppen und die Varianz innerhalb der Gruppen.
# **ANCOVA** (Analysis of Covariance) ist eine Erweiterung der ANOVA, die eine oder mehrere Kovariaten (kontinuierliche Variablen) berücksichtigt. Dies hilft, die Variabilität innerhalb der Gruppen zu reduzieren und die Teststärke für die Gruppeneffekte zu erhöhen, indem der Einfluss der Kovariaten statistisch kontrolliert wird.

# %% [code]
print("\n--- 3. Varianzanalyse (ANOVA) und Kovarianzanalyse (ANCOVA) ---")

# %% [markdown]
# ### 3.1. Daten für ANOVA/ANCOVA aus dem Advertising-Datensatz ableiten
# Um ANOVA und ANCOVA anzuwenden, müssen wir kategoriale Gruppen erstellen. Wir leiten eine kategoriale Variable `Strategy` ab, basierend auf dem `Radio`-Budget, und verwenden `Newspaper` als Kovariate (`Pre_Strategy_Budget`).

# %% [code]
# Erstellen einer Kopie des ursprünglichen DataFrames, um Manipulationen separat für die ANOVA/ANCOVA durchzuführen.
df_anova = df_marketing.copy()

# Erstellen der kategorialen Variable 'Strategy' aus 'Radio'-Ausgaben.
# `pd.qcut` teilt die Daten in `q` gleich große Quantile (hier 3 für 'Low', 'Medium', 'High').
# `labels` weist den Quantilen aussagekräftige Namen zu.
df_anova['Strategy'] = pd.qcut(df_anova['Radio'], q=3, labels=['Low Radio Budget', 'Medium Radio Budget', 'High Radio Budget'])

# Auswahl der relevanten Spalten und Umbenennung von 'Newspaper' in 'Pre_Strategy_Budget' für die Kovariate.
# Dies macht die Rolle der Variable in der ANCOVA-Modellierung klarer.
df_anova = df_anova[['Sales', 'Strategy', 'Newspaper']].rename(columns={'Newspaper': 'Pre_Strategy_Budget'})

print("\nErste 5 Zeilen der abgeleiteten ANOVA-Daten:")
print(df_anova.head())
print("\n")

# %% [markdown]
# ### 3.2. Durchführung der ANOVA
# Die ANOVA wird nun durchgeführt, um zu prüfen, ob es signifikante Unterschiede in den `Sales`-Mittelwerten zwischen den verschiedenen `Strategy`-Gruppen gibt.

# %% [code]
# Modellaufbau für ANOVA: `Sales ~ C(Strategy)`.
# `C(Strategy)` ist die `statsmodels`-Syntax, um anzuzeigen, dass `Strategy` als kategoriale Variable behandelt werden soll.
# `.fit()` führt das Modell aus.
model_anova = smf.ols('Sales ~ C(Strategy)', data=df_anova).fit()

# Erstellung der ANOVA-Tabelle mit `sm.stats.anova_lm`.
# `typ=2` gibt den Typ der Summe der Quadrate an. Typ 2 ist üblich bei unbalancierten Designs und bei Designs ohne Interaktionsterme.
anova_table = sm.stats.anova_lm(model_anova, typ=2)

print("ANOVA-Tabelle (Test auf Unterschiede zwischen den Strategien basierend auf Radio-Budget):")
print(anova_table)

# %% [markdown]
# #### Erklärung der ANOVA-Ergebnisse (F-Test)
# Der F-Test in der ANOVA prüft, ob die Mittelwerte der Gruppen statistisch signifikant voneinander abweichen.

# %% [code]
print("\n--- Erklärung der ANOVA-Ergebnisse ---")
print(f"  Der F-Wert für 'C(Strategy)' (hier: {anova_table['F']['C(Strategy)']: .2f}) und sein p-Wert (PR(>F): {anova_table['PR(>F)']['C(Strategy)']: .3f}) testen die Nullhypothese, dass die Mittelwerte der Verkäufe für alle Marketingstrategien (basierend auf Radio-Budget) gleich sind.")
print("  Ein kleiner p-Wert (typischerweise < 0.05) bedeutet, dass es statistisch signifikante Unterschiede zwischen den Mittelwerten der Gruppen gibt.")
print(f"  Da unser p-Wert ({anova_table['PR(>F)']['C(Strategy)']: .3f}) {'kleiner' if anova_table['PR(>F)']['C(Strategy)'] < 0.05 else 'größer oder gleich'} 0.05 ist, {'können wir die Nullhypothese verwerfen und schlussfolgern, dass es signifikante Unterschiede zwischen den Strategien gibt.' if anova_table['PR(>F)']['C(Strategy)'] < 0.05 else 'können wir die Nullhypothese nicht verwerfen, es gibt keine signifikanten Unterschiede zwischen den Strategien.'}")

# %% [markdown]
# ### 3.3. Post-hoc t-Tests nach signifikanter ANOVA (z.B. Tukey HSD)
# Wenn die ANOVA einen signifikanten Unterschied zwischen den Gruppenmittelwerten feststellt, wissen wir, DASS es Unterschiede gibt, aber nicht, WELCHE spezifischen Gruppen sich voneinander unterscheiden. Post-hoc-Tests wie der Tukey HSD (Honest Significant Difference) werden verwendet, um alle paarweisen Vergleiche zwischen den Gruppen durchzuführen und dabei für multiple Vergleiche zu korrigieren, um die Fehlerwahrscheinlichkeit zu kontrollieren.

# %% [code]
# Nur wenn die ANOVA signifikant war (p-Wert < 0.05) und die Variable 'C(Strategy)' in der Tabelle vorhanden ist,
# werden Post-hoc-Tests durchgeführt.
if 'C(Strategy)' in anova_table['PR(>F)'] and anova_table['PR(>F)']['C(Strategy)'] < 0.05:
    print("\n--- Post-hoc Tukey HSD Test (nach signifikanter ANOVA) ---")
    # `pairwise_tukeyhsd` führt den Tukey HSD Test durch.
    # `endog`: Die abhängige Variable (`Sales`).
    # `groups`: Die kategoriale Gruppierungsvariable (`Strategy`).
    # `alpha`: Das Signifikanzniveau für die Tests (meist 0.05).
    tukey_results = pairwise_tukeyhsd(endog=df_anova['Sales'], groups=df_anova['Strategy'], alpha=0.05)
    print(tukey_results)
    print("\nErklärung Post-hoc Tukey HSD Test:")
    print("  Die Tabelle zeigt für jedes Gruppenpaar den Unterschied der Mittelwerte ('meandiff'), das 95%-Konfidenzintervall für diesen Unterschied ('lower', 'upper'), den angepassten p-Wert ('p-adj') und ob der Unterschied statistisch signifikant ist ('reject').")
    print("  'reject=True' bedeutet, dass der Unterschied zwischen diesen beiden Gruppen statistisch signifikant ist (p-adj < 0.05), was auf einen echten Unterschied in den Verkaufszahlen hindeutet.")
else:
    print("\nANOVA war nicht signifikant oder 'C(Strategy)' nicht in der Tabelle, daher sind keine Post-hoc-Tests erforderlich.")

# %% [markdown]
# ### 3.4. Durchführung der Kovarianzanalyse (ANCOVA)
# Die ANCOVA erweitert die ANOVA, indem sie den Einfluss einer oder mehrerer kontinuierlicher Kovariaten statistisch kontrolliert. Hier fügen wir `Pre_Strategy_Budget` (Newspaper-Ausgaben) als Kovariate hinzu, um deren Einfluss auf die `Sales` zu berücksichtigen, während wir die Unterschiede zwischen den `Strategy`-Gruppen untersuchen.

# %% [code]
# Modellaufbau für ANCOVA: `Sales ~ C(Strategy) + Pre_Strategy_Budget`.
# Hier wird zusätzlich zur kategorialen Variable `Strategy` die kontinuierliche Variable `Pre_Strategy_Budget` hinzugefügt.
model_ancova = smf.ols('Sales ~ C(Strategy) + Pre_Strategy_Budget', data=df_anova).fit()

# Erstellung der ANCOVA-Tabelle. Typ 2 Summe der Quadrate wird erneut verwendet.
ancova_table = sm.stats.anova_lm(model_ancova, typ=2)

print("\nANCOVA-Tabelle (Test auf Unterschiede der Strategien unter Kontrolle des Pre-Strategie-Budgets (Newspaper)):")
print(ancova_table)

# %% [markdown]
# #### Erklärung der ANCOVA-Ergebnisse
# Die ANCOVA-Ergebnisse zeigen den Einfluss der Strategiegruppen und der Kovariate auf die Verkäufe.

# %% [code]
print("\n--- Erklärung der ANCOVA-Ergebnisse ---")
print("  Die ANCOVA testet weiterhin die Unterschiede zwischen den Marketingstrategien ('C(Strategy)'),")
print("  aber sie 'korrigiert' oder 'kontrolliert' für den Einfluss der Kovariate ('Pre_Strategy_Budget'). Dies bedeutet, dass die Effekte der Strategien nun unter der Annahme interpretiert werden, dass die 'Pre_Strategy_Budget' über alle Strategien hinweg konstant ist (oder die Vergleiche werden um die Unterschiede im 'Pre_Strategy_Budget' bereinigt).")
print(f"  Für 'C(Strategy)': F-Wert = {ancova_table['F']['C(Strategy)']: .2f}, p-Wert = {ancova_table['PR(>F)']['C(Strategy)']: .3f}.")
print(f"  Für 'Pre_Strategy_Budget': F-Wert = {ancova_table['F']['Pre_Strategy_Budget']: .2f}, p-Wert = {ancova_table['PR(>F)']['Pre_Strategy_Budget']: .3f}.")
print("  Ein signifikanter p-Wert für 'C(Strategy)' würde bedeuten, dass die Marketingstrategien auch nach Berücksichtigung des Budgets noch signifikante Auswirkungen auf die Verkäufe haben.")
print("  Ein signifikanter p-Wert für 'Pre_Strategy_Budget' würde bedeuten, dass das Budget ein wichtiger Prädiktor für die Verkäufe ist, unabhängig von der gewählten Strategie und den spezifischen Marketingstrategien.")

# %% [markdown]
# ## Optionale Visualisierung der Regressionsergebnisse
# Die folgenden Visualisierungen helfen dabei, die Anpassung des multiplen linearen Regressionsmodells zu bewerten und die Modellannahmen zu überprüfen.

# %% [code]
print("\n--- Visualisierung der Regressionsergebnisse (Beispiel) ---")

# Vorhersagen des Modells abrufen.
# `model_mlr.predict(df_marketing)` berechnet die vorhergesagten `Sales`-Werte basierend auf dem trainierten Modell und den originalen unabhängigen Variablen.
df_marketing['Predicted_Sales'] = model_mlr.predict(df_marketing)

# #### Streudiagramm: Tatsächliche vs. Vorhergesagte Werte
# Dieses Diagramm zeigt die tatsächlichen `Sales`-Werte im Vergleich zu den vom Modell vorhergesagten Werten. Eine gute Modellpassung würde Punkte nahe der 45-Grad-Linie (rot gestrichelt) zeigen.
plt.figure(figsize=(10, 6))
sns.scatterplot(x='Sales', y='Predicted_Sales', data=df_marketing, alpha=0.6)
plt.plot([df_marketing['Sales'].min(), df_marketing['Sales'].max()],
         [df_marketing['Sales'].min(), df_marketing['Sales'].max()],
         'r--', lw=2) # 45-Grad-Linie als Referenz
plt.title('Tatsächliche vs. Vorhergesagte Verkaufszahlen (Multiple Lineare Regression)')
plt.xlabel('Tatsächliche Verkaufszahlen')
plt.ylabel('Vorhergesagte Verkaufszahlen')
plt.grid(True)
plt.show()

# #### Residuenplot zur Überprüfung der Homoskedastizität und Linearität
# Ein Residuenplot zeigt die Residuen (Fehler) gegen die vorhergesagten Werte. Bei einem gut angepassten Modell ohne Verletzung der Annahmen sollten die Residuen zufällig um Null streuen, ohne erkennbare Muster (was auf Homoskedastizität und Linearität hindeutet).
plt.figure(figsize=(10, 6))
sns.scatterplot(x=model_mlr.fittedvalues, y=model_mlr.resid, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--') # Nulllinie für Residuen
plt.title('Residuenplot (Multiple Lineare Regression)')
plt.xlabel('Vorhergesagte Werte')
plt.ylabel('Residuen')
plt.grid(True)
plt.show()

# #### Histogramm der Residuen zur Überprüfung der Normalverteilung
# Dieses Histogramm zeigt die Verteilung der Residuen. Für die Gültigkeit vieler statistischer Tests (insbesondere der p-Werte in der Regression) wird angenommen, dass die Residuen normalverteilt sind (glockenförmige Verteilung).
plt.figure(figsize=(10, 6))
sns.histplot(model_mlr.resid, kde=True) # `kde=True` zeigt eine Kerndichteschätzung der Verteilung
plt.title('Histogramm der Residuen (Multiple Lineare Regression)')
plt.xlabel('Residuen')
plt.ylabel('Häufigkeit')
plt.show()

# %% [markdown]
# ## Überprüfung der Modellannahmen
# Dieser Abschnitt fasst die visuellen Checks und einen statistischen Test zur Überprüfung der Annahmen der linearen Regression zusammen.

# %% [code]
print("\n--- Überprüfung der Modellannahmen ---")
print("  Die obigen Visualisierungen helfen bei der Überprüfung der Annahmen der linearen Regression:")
print("  1. Linearität: Der Streudiagramm 'Tatsächliche vs. Vorhergesagte Verkaufszahlen' sollte eine lineare Beziehung entlang der 45-Grad-Linie zeigen.")
print("  2. Homoskedastizität (gleichbleibende Varianz der Fehler): Der Residuenplot sollte keine erkennbaren Muster oder sich verändernde Streuung der Punkte zeigen, die auf eine ungleichmäßige Varianz hindeuten.")
print("  3. Normalverteilung der Residuen: Das Histogramm der Residuen sollte annähernd normalverteilt sein (glockenförmig ohne starke Schiefe oder Ausreißer).")
print("  4. Unabhängigkeit der Residuen: Dies kann nicht direkt aus den Plots abgelesen werden, ist aber bei Zeitreihendaten oft ein Problem. Hier wurde es angenommen, da keine Zeitreihendaten vorliegen.")

# #### Shapiro-Wilk Test auf Normalität der Residuen
# Der Shapiro-Wilk Test ist ein statistischer Hypothesentest, der prüft, ob eine Stichprobe (hier: die Residuen) aus einer normalverteilten Grundgesamtheit stammt. Die Nullhypothese ist, dass die Daten normalverteilt sind.
shapiro_test = stats.shapiro(model_mlr.resid)
print(f"\nShapiro-Wilk Test auf Normalität der Residuen: Statistik={shapiro_test.statistic:.3f}, p-Wert={shapiro_test.pvalue:.3f}")
if shapiro_test.pvalue < 0.05:
    print("  Der p-Wert ist < 0.05, daher wird die Nullhypothese der Normalverteilung der Residuen verworfen. Dies deutet darauf hin, dass die Residuen wahrscheinlich nicht normalverteilt sind.")
else:
    print("  Der p-Wert ist >= 0.05, daher kann die Nullhypothese der Normalverteilung der Residuen nicht verworfen werden. Dies deutet darauf hin, dass die Residuen wahrscheinlich normalverteilt sind.")
